# 06 - Pooled-Source Training (improved model)

Changes vs 05, all keeping the cross-project protocol (the target project's data never enters
training):
1. **Pooled sources** - instead of one source project, each target model trains on the combined
   train pools of ALL other projects (GPT2SP-style transfer; Fu & Tantithamthavorn 2023).
2. **Log target** - the network learns log1p(story point); predictions are mapped back with expm1.
   Story points are heavily skewed, and L1 on the log scale relativeizes the errors.
3. **Seed ensemble** - 3 models with different seeds; the mean of their predictions is the final
   prediction.

Validation: one global 5-fold CV on the pooled train data (reported separately from test, as
required). The number of epochs is fixed from the folds, then per target 3 models are trained on
the full pooled set.

Same seed-42 80:20 partitions as the baseline and 05, so all results are directly comparable.

In [ ]:
import glob, os, re, time
import numpy as np, pandas as pd

csv = glob.glob("/kaggle/input/**/all_issues_sp.csv", recursive=True)[0]
df = pd.read_csv(csv)
df["storypoint"] = pd.to_numeric(df["storypoint"], errors="coerce")
df = df.dropna(subset=["storypoint"])
df = df[(df.storypoint > 0) & (df.storypoint <= 40)]
df["text"] = df["text"].fillna("").astype(str)
df["project"] = df["project"].astype(str).str.strip()
df["pos"] = np.arange(len(df))           # position in the filtered frame

# identical split construction to 02/05 (seed 42), as local row ranks per project
rng = np.random.default_rng(42)
splits = {}
for proj, g in df.groupby("project"):
    idx = g.index.to_numpy()
    rng.shuffle(idx)
    n_test = max(1, int(round(0.2 * len(idx))))
    local = np.arange(len(g))
    lmap = dict(zip(g.index.to_numpy().tolist(), local.tolist()))
    splits[proj] = dict(test=np.array([lmap[v] for v in idx[:n_test]]),
                        train=np.array([lmap[v] for v in idx[n_test:]]))

EMB = glob.glob("/kaggle/input/**/emb", recursive=True)[0]
META = EMB.replace("/emb", "/meta")
def safe(n):
    return re.sub(r"[^A-Za-z0-9_]", "_", str(n))
_files = sorted(os.path.basename(p)[:-4] for p in glob.glob(f"{EMB}/*.npy"))
def resolve(name):
    c = safe(name)
    if c in _files:
        return c
    m = [f for f in _files if f.rstrip("_") == c.rstrip("_")]
    assert m, f"no embedding file for {name!r}"
    return m[0]

# load every project's embeddings once into one big fp16 array (global-pos indexed)
names = sorted(df.project.unique())
N, L, D = len(df), 128, 768
BIG = np.zeros((N, L, D), dtype=np.float16)
Y = df.storypoint.values.astype(np.float32)
NTK = np.zeros(N, dtype=np.int32)
POS = {}
for name in names:
    key = resolve(name)
    e = np.load(f"{EMB}/{key}.npy", mmap_mode="r")
    m = pd.read_csv(f"{META}/{key}.csv")
    g = df[df.project == name]
    assert len(e) == len(g) == len(m)
    gp = g.pos.values
    BIG[gp] = np.asarray(e)
    NTK[gp] = np.minimum(m.n_tokens.values, L)
    POS[name] = gp
    del e
print("pooled array:", BIG.shape, f"({BIG.nbytes/1e9:.1f} GB) |", len(names), "projects")

# global train pool = every project's 80% (exactly what per-pair training used, now combined)
train_all = np.concatenate([POS[p][splits[p]["train"]] for p in names])
print("pooled train size:", len(train_all))

base = pd.read_csv(glob.glob("/kaggle/input/**/baseline_results.csv", recursive=True)[0])
base["source"] = base["source"].astype(str).str.strip()
base["target"] = base["target"].astype(str).str.strip()

In [ ]:
import torch, torch.nn as nn
from sklearn.model_selection import KFold

MAX_LEN, D = 128, 768

class CNNRegressor(nn.Module):
    def __init__(self, d_in=D, ch=(256, 128, 64), p=0.2):
        super().__init__()
        c1, c2, c3 = ch
        self.conv = nn.Sequential(
            nn.Conv1d(d_in, c1, 3, padding=1), nn.ReLU(), nn.Dropout(p),
            nn.Conv1d(c1, c2, 3, padding=1), nn.ReLU(),
            nn.Conv1d(c2, c3, 3, padding=1), nn.ReLU(),
        )
        self.head = nn.Sequential(nn.Linear(c3, 32), nn.ReLU(), nn.Linear(32, 1))
    def forward(self, x, ntok):
        h = self.conv(x.transpose(1, 2))
        mask = (torch.arange(x.size(1), device=x.device)[None, :] < ntok[:, None]).unsqueeze(1)
        h = h.masked_fill(~mask, float("-inf"))
        return self.head(h.max(dim=2).values).squeeze(-1)

def batches(idx, bs, seed):
    perm = np.random.RandomState(seed).permutation(idx)
    for b0 in range(0, len(perm), bs):
        yield perm[b0:b0 + bs]

def train_one(idx_pool, ylog, epochs, bs, seed, Xva=None, yva=None, nva=None):
    torch.manual_seed(seed); np.random.seed(seed)
    model = CNNRegressor().cuda()
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    lossf = nn.L1Loss()
    best = (1e9, None, 0); hist = []
    for ep in range(epochs):
        model.train()
        for sel in batches(idx_pool, bs, seed * 1000 + ep):
            x = torch.tensor(np.asarray(BIG[sel]), dtype=torch.float32, device="cuda")
            nt = torch.tensor(NTK[sel], dtype=torch.long, device="cuda")
            y = torch.tensor(ylog[sel], dtype=torch.float32, device="cuda")
            with torch.autocast("cuda", dtype=torch.float16):
                loss = lossf(model(x, nt), y)
            opt.zero_grad(); loss.backward(); opt.step()
        if Xva is not None:
            model.eval(); errs = []
            with torch.no_grad():
                for b0 in range(0, len(yva), 512):
                    sel = Xva[b0:b0+512]
                    x = torch.tensor(np.asarray(BIG[sel]), dtype=torch.float32, device="cuda")
                    nt = torch.tensor(NTK[sel], dtype=torch.long, device="cuda")
                    yv = torch.tensor(yva[b0:b0+512], dtype=torch.float32, device="cuda")
                    p = torch.expm1(model(x, nt)).clamp(min=0).cpu().numpy()
                    errs.append(np.abs(p - np.expm1(yva[b0:b0+512])))
            vmae = float(np.concatenate(errs).mean()); hist.append(vmae)
            if vmae < best[0]: best = (vmae, {k: t.clone() for k, t in model.state_dict().items()}, ep)
            if ep - best[2] >= 4: break
    if Xva is not None: model.load_state_dict(best[1])
    return model, hist

def predict(model, idx, bs=512):
    model.eval(); out = []
    with torch.no_grad():
        for b0 in range(0, len(idx), bs):
            sel = idx[b0:b0+bs]
            x = torch.tensor(np.asarray(BIG[sel]), dtype=torch.float32, device="cuda")
            nt = torch.tensor(NTK[sel], dtype=torch.long, device="cuda")
            out.append(torch.expm1(model(x, nt)).clamp(min=0).cpu().numpy())
    return np.concatenate(out)

print("model ready")

In [ ]:
# --- global 5-fold CV on the pooled train data (one seed) -> validation stats + epoch choice ---
ylog = np.log1p(Y)
kf = KFold(n_splits=5, shuffle=True, random_state=42)
fold_maes, best_eps = [], []
tr_all = train_all.copy()
for f, (a, b) in enumerate(kf.split(tr_all)):
    va, tr = tr_all[b], tr_all[a]
    ntr = len(tr)
    tr_sub = tr if ntr <= 30000 else np.random.RandomState(42).choice(tr, 30000, replace=False)
    mdl, hist = train_one(tr_sub, ylog, epochs=6, bs=64, seed=42 + f, Xva=va, yva=ylog[va], nva=None)
    # validation batches need NTK but train_one's val loop indexes BIG only -> pass va indices
    fold_maes.append(hist[-1]); best_eps.append(len(hist))
    print(f"fold {f+1}: val MAE {hist[-1]:.3f} (stopped ep {len(hist)})", flush=True)
val_mean, val_std = float(np.mean(fold_maes)), float(np.std(fold_maes))
FIXED_EPOCHS = int(np.clip(round(np.mean(best_eps)), 2, 6))
print(f"pooled 5-fold val MAE: {val_mean:.3f} +/- {val_std:.3f} | fixed epochs: {FIXED_EPOCHS}")

In [ ]:
# --- per target: pooled-source 3-seed ensemble, evaluate on held-out test ---
PAIRS = [
    ("XD", "MESOS"), ("Usergrid", "MXNet"), ("Titanium Mobile Platform", "Usergrid"),
    ("The Titanium SDK", "Alloy Framework"), ("Software Cloud", "Appcelerator Daemon"),
    ("Sawtooth", "Appcelerator Studio"), ("Nexus", "Command-Line Interface"),
    ("MXNet", "The Titanium SDK"), ("Mule APIkit", "Titanium Mobile Platform"),
    ("Mule", "C++ driver (Appcelerator)"), ("Moodle", "Atlassian Software Server"),
    ("MongoDB Core Server", "Bamboo"), ("MESOS", "Clover"),
    ("Lyrasis Dura Cloud", "Confluence Cloud"), ("Lsstcorp Data management", "Confluence Server"),
    ("Jira Server", "Crowd"), ("Jira Cloud", "FishEye"), ("Java driver", "Jira Cloud"),
    ("Indy SDK", "Jira Server"), ("Indy Node", "Software Cloud"),
    ("FishEye", "DotNetNuke Platform"), ("Fabric", "Blockchain Explorer"),
    ("Evergreen", "Fabric"), ("DotNetNuke Platform", "Indy Node"), ("DataCass", "Indy SDK"),
    ("Crowd", "Sawtooth"), ("Confluence Server", "Lsstcorp Data management"),
    ("Confluence Cloud", "Lyrasis Dura Cloud"), ("Compass", "MongoDB Core Server"),
    ("Command-Line Interface", "Compass"), ("Clover", "C++ driver (MongoDB)"),
    ("C++ driver (Appcelerator)", "Evergreen"), ("C++ driver (MongoDB)", "Java driver"),
    ("Blockchain Explorer", "Moodle"), ("Bamboo", "Mule"),
    ("Atlassian Software Server", "Mule APIkit"), ("Appcelerator Studio", "Nexus"),
    ("Appcelerator Daemon", "DataCass"), ("Alloy Framework", "XD"),
]
NAME = {
    "XD": "Spring XD", "MESOS": "Apache Mesos", "Usergrid": "Apache Usergrid", "MXNet": "Apache MXNet",
    "The Titanium SDK": "The Titanium SDK", "Software Cloud": "Atlassian Software Cloud",
    "Sawtooth": "Hyperledger Sawtooth", "Nexus": "Sonatype Nexus", "Bamboo": "Atlassian Bamboo",
    "Clover": "Atlassian Clover", "Confluence Cloud": "Atlassian Confluence Cloud",
    "Confluence Server": "Atlassian Confluence Server", "Jira Server": "Atlassian Jira Server",
    "Crowd": "Atlassian Crowd", "Jira Cloud": "Atlassian Jira Cloud", "FishEye": "Atlassian FishEye",
    "Java driver": "The Mongo Java driver", "Indy SDK": "Hyperledger Indy SDK",
    "Indy Node": "Hyperledger Indy Node", "Fabric": "Hyperledger Fabric",
    "Blockchain Explorer": "Hyperledger Blockchain Explorer", "DataCass": "Spring DataCass",
    "Compass": "MongoDB Compass", "C++ driver (MongoDB)": "The Mongo C++ driver",
    "Evergreen": "The MongoDB Engineering", "C++ driver (Appcelerator)": "Aptana Studio",
    "Atlassian Software Server": "Atlassian Atlassian Software Server",
}
def dsn(n):
    return NAME.get(n, n)

# per target: pooled = train pools of every OTHER project (global positions)
pool_by_target = {}
for t in names:
    pool_by_target[t] = np.concatenate([POS[p][splits[p]["train"]] for p in names if p != t])

base_idx = base.set_index(["source", "target"])
rows = []
t0 = time.time()
for i, (s, t) in enumerate(PAIRS):
    src, tgt = dsn(s), dsn(t)
    te_pos = POS[tgt][splits[tgt]["test"]]
    preds = []
    for sd in (42, 43, 44):
        mdl, _ = train_one(pool_by_target[tgt], ylog, epochs=FIXED_EPOCHS, bs=64, seed=sd)
        preds.append(predict(mdl, te_pos))
        del mdl
        torch.cuda.empty_cache()
    pred = np.mean(preds, axis=0)
    ytest = Y[te_pos]
    err = np.abs(pred - ytest)
    brow = base_idx.loc[(src, tgt)]
    rows.append(dict(pdf_source=s, pdf_target=t, source=src, target=tgt,
                     test_mae=float(err.mean()), test_mdae=float(np.median(err)),
                     baseline_mae=float(brow.mae), baseline_mdae=float(brow.mdae),
                     epochs=FIXED_EPOCHS))
    pd.DataFrame(rows).to_csv("/kaggle/working/results_pooled_pairs.csv", index=False)
    print(f"[{i+1}/39] {s} -> {t}: MAE {rows[-1]['test_mae']:.3f} (base {rows[-1]['baseline_mae']:.3f}) | {time.time()-t0:.0f}s", flush=True)
print("ALL DONE in", round((time.time()-t0)/60, 1), "min")

In [ ]:
res = pd.DataFrame(rows)
tmae, bmae = res.test_mae, res.baseline_mae
lines = [
    "PS-3 cross-project evaluation - POOLED-source BERT+CNN (log target, 3-seed ensemble)",
    f"pooled 5-fold val MAE: {val_mean:.3f} +/- {val_std:.3f} | epochs {FIXED_EPOCHS} | seed 42 partitions",
    "",
    f"test MAE  across pairs: {tmae.mean():.3f} +/- {tmae.std():.3f}  (baseline {bmae.mean():.3f} +/- {bmae.std():.3f})",
    f"test MdAE across pairs: {res.test_mdae.mean():.3f}              (baseline {res.baseline_mdae.mean():.3f})",
    f"beats baseline MAE on {int((tmae < bmae).sum())}/{len(res)} pairs",
]
txt = chr(10).join(lines)
open("/kaggle/working/evaluation_summary_pooled.txt", "w").write(txt)
print(txt)

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(19, 5))
x = np.arange(len(res))
ax.bar(x - 0.2, bmae, 0.4, label="median baseline")
ax.bar(x + 0.2, tmae, 0.4, label="pooled BERT + CNN")
ax.set_xticks(x)
ax.set_xticklabels([f"{s[:9]}>{t[:9]}" for s, t in zip(res.pdf_source, res.pdf_target)], rotation=90, fontsize=6)
ax.axhline(bmae.mean(), color="gray", ls=":")
ax.set_ylabel("test MAE"); ax.legend()
ax.set_title("Pooled-source model vs baseline, test MAE per pair")
plt.tight_layout(); plt.savefig("/kaggle/working/results_pooled_per_pair.png", dpi=130); plt.show()